# Week 1 — Baseline Inference

Loads **LLaVA-1.5-7B** (pretrained, zero-shot) and runs it over the NLM-CXR test set  
to establish the pre-fine-tuning benchmark.

| Step | What happens |
|---|---|
| 1 | Load dataset splits from `cxr_dataset.csv` |
| 2 | Load model in 4-bit (BitsAndBytes) |
| 3 | Generate reports for `N_EVAL` test samples |
| 4 | Score with BLEU, ROUGE, BERTScore |
| 5 | Save predictions + metrics to `results/baseline/` |

In [ ]:
# Install if running on a fresh Colab instance
# !pip install -q transformers peft bitsandbytes accelerate trl \
#              nltk rouge-score bert-score scikit-learn wandb

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    LlavaForConditionalGeneration,
)

# Allow importing from src/
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dataset import load_splits, INSTRUCTION, EVAL_TEMPLATE
from src.metrics import evaluate_all, print_metrics

## Configuration

In [ ]:
MODEL_ID   = "llava-hf/llava-1.5-7b-hf"   # swap to llava-hf/llava-1.5-13b-hf for 13B
CSV_PATH   = ROOT / "cxr_dataset.csv"
OUTPUT_DIR = ROOT / "results" / "baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_EVAL        = 200   # number of test samples to evaluate
MAX_NEW_TOKENS = 256  # max tokens generated per report
SEED          = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load dataset

In [ ]:
train_df, val_df, test_df = load_splits(CSV_PATH, seed=SEED)

print(f"Train: {len(train_df):,} reports")
print(f"Val  : {len(val_df):,} reports")
print(f"Test : {len(test_df):,} reports")

# Sub-sample for faster baseline evaluation
eval_df = test_df.sample(min(N_EVAL, len(test_df)), random_state=SEED).reset_index(drop=True)
print(f"\nEvaluating on {len(eval_df)} test samples")

## 2. Load model (4-bit quantized)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model loaded.")

## 3. Inference loop

In [ ]:
from PIL import Image

hypotheses = []
references = []
failures   = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Inference"):
    try:
        image = Image.open(row["primary_image"]).convert("RGB")
        prompt = EVAL_TEMPLATE.format(instruction=INSTRUCTION)

        inputs = processor(
            text=prompt,
            images=image,
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
            )

        # Strip the prompt tokens; decode only the newly generated tokens
        gen_ids = output_ids[0, inputs["input_ids"].shape[1]:]
        generated = processor.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

        hypotheses.append(generated)
        references.append(row["report_text"])

    except Exception as e:
        failures.append({"index": _, "error": str(e)})

print(f"\nGenerated {len(hypotheses)} reports, {len(failures)} failures")

## 4. Compute baseline metrics

In [ ]:
print("Computing metrics (BERTScore may take a minute)...")
metrics = evaluate_all(hypotheses, references)

print("\n=== Baseline Metrics (LLaVA-1.5-7B, zero-shot) ===")
print_metrics(metrics)

## 5. Inspect examples

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image as PILImage

N_SHOW = 3

for i in range(min(N_SHOW, len(hypotheses))):
    row = eval_df.iloc[i]
    img = PILImage.open(row["primary_image"]).convert("RGB")

    fig, (ax_img, ax_text) = plt.subplots(1, 2, figsize=(14, 5))
    ax_img.imshow(img, cmap="gray")
    ax_img.axis("off")
    ax_img.set_title(f"Sample {i+1}  —  {row.get('uid', row['report_id'])}")

    text = (
        f"REFERENCE:\n{references[i]}\n\n"
        f"{'─'*60}\n\n"
        f"GENERATED:\n{hypotheses[i]}"
    )
    ax_text.text(
        0, 1, text,
        va="top", ha="left",
        fontsize=8, wrap=True,
        transform=ax_text.transAxes,
        fontfamily="monospace",
    )
    ax_text.axis("off")
    plt.tight_layout()
    plt.show()

## 6. Save results

In [ ]:
# Metrics JSON
metrics_path = OUTPUT_DIR / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump({"model": MODEL_ID, "n_eval": len(hypotheses), **metrics}, f, indent=2)
print(f"Metrics → {metrics_path}")

# Predictions CSV
preds_path = OUTPUT_DIR / "predictions.csv"
pred_df = eval_df[["report_id", "uid", "primary_image", "report_text"]].copy()
pred_df["generated"] = hypotheses
pred_df.to_csv(preds_path, index=False)
print(f"Predictions → {preds_path}")